In [5]:
import numpy as np
import pandas as pd


In [5]:
def download():
    
    from urllib.request import urlretrieve
    import os

    url = (
        "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/"
        "Maji_Ndogo/Maji_Ndogo_farm_survey_small.db"
    )

    db_file = "Maji_Ndogo_farm_survey_small.db"

    # Download only if the database is not already present
    if not os.path.exists(db_file):
        urlretrieve(url, db_file)
        print(f"Downloaded '{db_file}'.")
    else:
        print(f"'{db_file}' already exists.")

    return

In [3]:
def data_base():
    import pandas as pd
    from sqlalchemy import create_engine, text

    engine = create_engine('sqlite:///Maji_Ndogo_farm_survey_small.db')

    sql_query = """
    SELECT *
    FROM geographic_features
    LEFT JOIN weather_features USING (Field_ID)
    LEFT JOIN soil_and_crop_features USING (Field_ID)
    LEFT JOIN farm_management_features USING (Field_ID)
    """

    with engine.connect() as connection:
        MD_agric_df = pd.read_sql_query(text(sql_query), connection)

    # Wrangling: the 'Crop_type' and 'Annual_yield' labels were swapped on export, so swap them back;
    # elevations were stored with the wrong sign; and a few crop names were misspelt.
    MD_agric_df.rename(columns={'Annual_yield': 'Crop_type_Temp', 'Crop_type': 'Annual_yield'}, inplace=True)
    MD_agric_df.rename(columns={'Crop_type_Temp': 'Crop_type'}, inplace=True)
    MD_agric_df['Elevation'] = MD_agric_df['Elevation'].abs()
    corrections = {'cassaval': 'cassava', 'wheatn': 'wheat', 'teaa': 'tea'}
    MD_agric_df['Crop_type'] = MD_agric_df['Crop_type'].apply(
        lambda crop: corrections.get(crop.strip(), crop.strip())
    )
    print(f'Loaded: {MD_agric_df.shape[0]} rows × {MD_agric_df.shape[1]} columns')
    return (MD_agric_df,)

In [4]:
MD_agric_df = data_base()[0]
print(type(MD_agric_df))

Loaded: 5654 rows × 18 columns
<class 'pandas.DataFrame'>


In [6]:
print(MD_agric_df.columns.tolist())

['Field_ID', 'Elevation', 'Latitude', 'Longitude', 'Location', 'Slope', 'Rainfall', 'Min_temperature_C', 'Max_temperature_C', 'Ave_temps', 'Soil_fertility', 'Soil_type', 'pH', 'Pollution_level', 'Plot_size', 'Annual_yield', 'Crop_type', 'Standard_yield']


### Data dictionary

**Geographic features**
| Column | Description | Type |
|--------|-------------|------|
| Field_ID | Unique identifier for each field | BigInt |
| Elevation | Elevation above sea level in meters | Float |
| Latitude / Longitude | Geographic coordinates in degrees | Float |
| Location | Province the field is in | Text |
| Slope | Slope of the land in degrees | Float |

**Weather features**
| Column | Description | Type |
|--------|-------------|------|
| Rainfall | Annual rainfall in mm | Float |
| Min_temperature_C | Average minimum temperature in °C | Float |
| Max_temperature_C | Average maximum temperature in °C | Float |
| Ave_temps | Average temperature in °C — mean of Min and Max | Float |

**Soil features**
| Column | Description | Type |
|--------|-------------|------|
| Soil_fertility | Normalized fertility index, 0–1 | Float |
| Soil_type | Categorical soil classification | Text |
| pH | Soil pH level | Float |

**Farm management features**
| Column | Description | Type |
|--------|-------------|------|
| Pollution_level | Normalized pollution index, 0–1 | Float |
| Plot_size | Field area in hectares | Float |
| Crop_type | Type of crop grown | Text |
| Annual_yield | Total annual yield in tonnes | Float |
| Standard_yield | Normalized yield score, 0–1 | Float |

## __Challenge 1: Crop distribution__

Before recommending where to plant, we need to understand where crops currently grow and what conditions they share. For a given crop type, the average rainfall and average elevation immediately tells us whether it is a highland or lowland species.

### __Task__

Complete `explore_crop_distribution(df, crop_filter)`. It must filter the DataFrame to rows where `Crop_type` matches `crop_filter`, then return a **tuple** of `(mean Rainfall, mean Elevation)` as plain Python floats.

> ⚠️ Do not change the function name `explore_crop_distribution`.

### __Expected output__

> **Input 1:** `explore_crop_distribution(MD_agric_df, 'tea')` → `(1534.5079956188388, 775.208667535597)`

> **Input 2:** `explore_crop_distribution(MD_agric_df, 'wheat')` → `(1010.2859910581222, 595.8384148002981)`

In [7]:
def explore_crop_distribution(df, crop_filter):
    """
    Returns the mean rainfall and mean elevation
    for the specified crop type.

    Parameters
    ----------
    df : pandas.DataFrame
        Agricultural dataset.
    crop_filter : str
        Crop type to filter (e.g., 'tea', 'wheat').

    Returns
    -------
    tuple
        (mean_rainfall, mean_elevation)
    """

    filtered_df = df[df["Crop_type"] == crop_filter]

    mean_rainfall = float(filtered_df["Rainfall"].mean())
    mean_elevation = float(filtered_df["Elevation"].mean())

    return (mean_rainfall, mean_elevation)

In [8]:
tea = explore_crop_distribution(MD_agric_df, 'tea')
print(tea)

(1534.5079956188388, 775.208667535597)


In [9]:
wheat = explore_crop_distribution(MD_agric_df, 'wheat')
print(wheat)

(1010.2859910581222, 595.8384148002981)


In [10]:
print(tea[1] - wheat[1])

179.3702527352989


## __Challenge 2: Soil fertility by type__

Soil type is one of the strongest predictors of crop success.

### __Task__

Complete `analyse_soil_fertility(df)`. It must group the DataFrame by `Soil_type`, calculate the mean `Soil_fertility` per group, and return the result as a **Pandas Series**.

> ⚠️ Do not change the function name `analyse_soil_fertility`.

### __Expected output__

    ```
    Soil_type
    Loamy       0.585868
    Peaty       0.604882
    Rocky       0.582368
    Sandy       0.595669
    Silt        0.652654
    Volcanic    0.648894
    Name: Soil_fertility, dtype: float64

In [14]:
def analyse_soil_fertility(df):
    """
    Groups the data by soil type and returns the
    average soil fertility for each soil type.

    Parameters
    ----------
    df : pandas.DataFrame
        Agricultural dataset.

    Returns
    -------
    pandas.Series
        Mean soil fertility for each soil type.
    """
    return df.groupby("Soil_type")["Soil_fertility"].mean()

In [15]:
soil_fertility = analyse_soil_fertility(MD_agric_df)
print(soil_fertility)

Soil_type
Loamy       0.585868
Peaty       0.604882
Rocky       0.582368
Sandy       0.595669
Silt        0.652654
Volcanic    0.648894
Name: Soil_fertility, dtype: float64


## __Challenge 3: Climate and geography by crop__

For every crop type, we need the average elevation, minimum temperature, maximum temperature, and rainfall.

### __Task__

Complete `climate_geography_influence(df, column)`. It must group the DataFrame by `column`, calculate the mean of `Elevation`, `Min_temperature_C`, `Max_temperature_C`, and `Rainfall` — **in that order** — and return the result as a **DataFrame**.

> ⚠️ Do not change the function name `climate_geography_influence`.

### __Expected output__

    ```
                 Elevation  Min_temperature_C  Max_temperature_C    Rainfall
    Crop_type
    banana      487.973572          -5.354344          31.988152  1659.905687
    cassava     682.903008          -3.992113          30.902381  1210.543006
    coffee      647.047734          -4.028007          30.855189  1527.265074
    maize       680.596982          -4.497995          30.576692   681.010276
    potato      696.313917          -4.375334          30.300608   660.289064
    rice        352.858053          -6.610566          32.727170  1632.382642
    tea         775.208668          -2.862651          29.950383  1534.507996
    wheat       595.838415          -4.968107          30.973845  1010.285991
    ```

In [11]:
def climate_geography_influence(df, column):
    """
    Groups the data by the specified column and returns the
    mean elevation, minimum temperature, maximum temperature,
    and rainfall.

    Parameters
    ----------
    df : pandas.DataFrame
        Agricultural dataset.
    column : str
        Column to group by (e.g., 'Crop_type').

    Returns
    -------
    pandas.DataFrame
        Mean Elevation, Min_temperature_C,
        Max_temperature_C, and Rainfall.
    """
    return (
        df.groupby(column)[
            [
                "Elevation",
                "Min_temperature_C",
                "Max_temperature_C",
                "Rainfall",
            ]
        ]
        .mean()
        .sort_values("Elevation", ascending = False)
    )

In [12]:
result = climate_geography_influence(MD_agric_df, "Crop_type")
print(result)

            Elevation  Min_temperature_C  Max_temperature_C     Rainfall
Crop_type                                                               
tea        775.208668          -2.862651          29.950383  1534.507996
potato     696.313917          -4.375334          30.300608   660.289064
cassava    682.903008          -3.992113          30.902381  1210.543006
maize      680.596982          -4.497995          30.576692   681.010276
coffee     647.047734          -4.028007          30.855189  1527.265074
wheat      595.838415          -4.968107          30.973845  1010.285991
banana     487.973572          -5.354344          31.988152  1659.905687
rice       352.858053          -6.610566          32.727170  1632.382642


In [13]:
soil = climate_geography_influence(MD_agric_df, "Soil_type")
print(soil)

            Elevation  Min_temperature_C  Max_temperature_C     Rainfall
Soil_type                                                               
Rocky      892.665740          -2.425658          29.131579   841.874671
Volcanic   750.902092          -2.993755          30.089992  1630.504364
Sandy      743.456509          -3.821689          30.175496   797.665003
Loamy      552.383554          -5.620966          31.374717   724.785612
Peaty      467.291922          -5.835294          32.032941  1344.381176
Silt       424.196238          -5.927452          32.306236  1667.228365


## __Challenge 4: Top-performing crop__

Time to identify Maji Ndogo's single best-performing crop — the one with the most fields exceeding the average `Standard_yield`.

**Hint:** After grouping, the labels of the grouping column can be accessed with `.index`. For example:
```python
    grouped_df = MD_agric_df.groupby('Soil_type').mean(numeric_only=True).sort_values('Elevation', ascending=False)
    print(grouped_df.index[0])
```

### __Task__

Complete `find_ideal_fields(df)`. It must:
1. Filter to fields with an above-average `Standard_yield`.
2. Group by `Crop_type` and count records per group.
3. Sort to put the highest count first.
4. Return the `Crop_type` name at index 0 as a **string**.

> ⚠️ Do not change the function name `find_ideal_fields`.

### __Expected output__

> **Input 1:** `type(find_ideal_fields(MD_agric_df))` → `<class 'str'>`

> **Input 2:** `print('Top-performing crop:', find_ideal_fields(MD_agric_df))` → `Top-performing crop: tea`

In [18]:
def find_ideal_fields(df):
    """
    Finds the crop type with the highest number of fields
    exceeding the average Standard_yield.

    Parameters
    ----------
    df : pandas.DataFrame
        Agricultural dataset.

    Returns
    -------
    str
        Crop type with the most above-average yielding fields.
    """
    
    average_yield = df["Standard_yield"].mean()

    above_average = df[
        df["Standard_yield"] > average_yield
    ]

    crop_counts = (
        above_average
        .groupby("Crop_type")
        .size()
        .sort_values(ascending=False)
    )

    return str(crop_counts.index[0])

#### __Code Explanation__
__Why .size() is used__

__After filtering:__

``` python
above_average.groupby("Crop_type")

```
> creates groups for each crop.
> `.size()` counts the number of rows (fields) in each crop group:

Then:

``` python
.sort_values(ascending=False)

```

> places the crop with the most high-performing fields at the top.

> __Finally:__

``` python
crop_counts.index[0]

```

> returns the name of the first crop `("tea")`, and `str()` ensures the output is a Python string as required.

In [21]:
print(type(find_ideal_fields(MD_agric_df)))

<class 'str'>


In [20]:
print('Top-performing crop:', find_ideal_fields(MD_agric_df))

Top-performing crop: tea


 ## __Challenge 5: Ideal growing conditions__

Given a crop type, return only the fields that meet all four quality criteria simultaneously. This is a natural fit for the `.query()` method, which reads almost like the sentence above — though a boolean mask works just as well.

### __Task__

Complete `find_good_conditions(df, crop_type)`. Filter to rows where:
1. `Crop_type` matches `crop_type`.
2. `Standard_yield` is above that crop's own mean `Standard_yield`.
3. `Ave_temps` is between 12 and 15 (inclusive).
4. `Pollution_level` is below 0.0001.

Return the filtered **DataFrame**.

> 📌 `Ave_temps` is pre-computed in the database as the average of `Min_temperature_C` and `Max_temperature_C`.

> ⚠️ Do not change the function name `find_good_conditions`.

### __Expected output__

> **Input 1:** `find_good_conditions(MD_agric_df, 'tea').shape` → `(14, 18)`

In [22]:
def find_good_conditions(df, crop_type):
    """
    Returns fields that satisfy ideal growing conditions
    for a given crop.

    Parameters
    ----------
    df : pandas.DataFrame
        Agricultural dataset.
    crop_type : str
        Crop to filter.

    Returns
    -------
    pandas.DataFrame
        Fields meeting all quality criteria.
    """

    crop_mean_yield = df.loc[
        df["Crop_type"] == crop_type,
        "Standard_yield"
    ].mean()

    good_conditions = df.query(
        "Crop_type == @crop_type and "
        "Standard_yield > @crop_mean_yield and "
        "Ave_temps >= 12 and "
        "Ave_temps <= 15 and "
        "Pollution_level < 0.0001"
    )

    return good_conditions

#### __Code Explanation__
<br>1. 
``` python
crop_mean_yield = df.loc[
    df["Crop_type"] == crop_type,
    "Standard_yield"
].mean()
```
> calculates the average yield for only that crop.
  
<br>2. Then this condition:
   
``` python
Standard_yield > @crop_mean_yield
```
- keeps only tea fields performing above the normal tea yield.
- The `@` symbol in `.query()` allows you to use a Python variable `(crop_type, crop_mean_yield)` inside the query expression.

In [23]:
tea_fields = find_good_conditions(MD_agric_df, "tea")

print(tea_fields.shape)

(14, 18)
